In [1]:
import pandas as pd
import numpy as np
import folium
from math import radians, sin, cos, sqrt, atan2
from IPython.display import display

print("Library berhasil diimpor.")

Library berhasil diimpor.


In [20]:
toko = pd.DataFrame({
    "nama_toko": ["Toko A", "Toko B", "Toko C", "Toko D"],
    "latitude": [-6.2000, -6.2150, -6.1900, -6.2250],
    "longitude": [106.8160, 106.8300, 106.8400, 106.8050]
})

kandidat = pd.DataFrame({
    "lokasi": ["Kandidat 1", "Kandidat 2", "Kandidat 3", "Kandidat 4", "Kandidat 5"],
    "latitude": [-6.2050, -6.2100, -6.1950, -6.2300, -6.1850],
    "longitude": [106.8500, 106.8100, 106.8200, 106.8350, 106.8000],
    "kepadatan_penduduk": [9200, 7600, 8800, 6400, 9500]
})

print("Data toko:")
display(toko)
print("Data kandidat lokasi:")
display(kandidat)

Data toko:


,nama_toko,latitude,longitude
0,Toko A,-6.200,106.816
1,Toko B,-6.215,106.830
2,Toko C,-6.190,106.840
3,Toko D,-6.225,106.805


Data kandidat lokasi:


,lokasi,latitude,longitude,kepadatan_penduduk
0,Kandidat 1,-6.205,106.850,9200
1,Kandidat 2,-6.210,106.810,7600
2,Kandidat 3,-6.195,106.820,8800
3,Kandidat 4,-6.230,106.835,6400
4,Kandidat 5,-6.185,106.800,9500


In [21]:
def jarak_km(lat1, lon1, lat2, lon2):
    radius_bumi = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    return 2 * radius_bumi * atan2(sqrt(a), sqrt(1 - a))

jarak_terdekat = []
for _, calon in kandidat.iterrows():
    jarak = toko.apply(
        lambda t: jarak_km(calon["latitude"], calon["longitude"],
                           t["latitude"], t["longitude"]),
        axis=1
    )
    jarak_terdekat.append(jarak.min())

kandidat["jarak_toko_terdekat_km"] = np.round(jarak_terdekat, 2)
display(kandidat)

,lokasi,latitude,longitude,kepadatan_penduduk,jarak_toko_terdekat_km
0,Kandidat 1,-6.205,106.850,9200,2.00
1,Kandidat 2,-6.210,106.810,7600,1.29
2,Kandidat 3,-6.195,106.820,8800,0.71
3,Kandidat 4,-6.230,106.835,6400,1.76
4,Kandidat 5,-6.185,106.800,9500,2.43


In [22]:
min_padat = kandidat["kepadatan_penduduk"].min()
max_padat = kandidat["kepadatan_penduduk"].max()
min_jarak = kandidat["jarak_toko_terdekat_km"].min()
max_jarak = kandidat["jarak_toko_terdekat_km"].max()

kandidat["skor_kepadatan"] = (
    (kandidat["kepadatan_penduduk"] - min_padat) / (max_padat - min_padat)
    if max_padat != min_padat else 0.5
)
kandidat["skor_jarak"] = (
    (kandidat["jarak_toko_terdekat_km"] - min_jarak) / (max_jarak - min_jarak)
    if max_jarak != min_jarak else 0.5
)
kandidat["skor_potensi"] = (
    0.6 * kandidat["skor_kepadatan"] + 0.4 * kandidat["skor_jarak"]
).round(3)

kandidat = kandidat.sort_values("skor_potensi", ascending=False).reset_index(drop=True)
kandidat["peringkat"] = kandidat.index + 1

display(kandidat[["peringkat", "lokasi", "kepadatan_penduduk",
                  "jarak_toko_terdekat_km", "skor_potensi"]])

,peringkat,lokasi,kepadatan_penduduk,jarak_toko_terdekat_km,skor_potensi
0,1,Kandidat 5,9500,2.43,1.000
1,2,Kandidat 1,9200,2.00,0.842
2,3,Kandidat 3,8800,0.71,0.465
3,4,Kandidat 2,7600,1.29,0.367
4,5,Kandidat 4,6400,1.76,0.244


In [10]:
!pip -q install geodatasets

In [23]:
import folium
import pandas as pd
from IPython.display import display, Markdown

peta = folium.Map(
    location=[
        kandidat["latitude"].mean(),
        kandidat["longitude"].mean()
    ],
    zoom_start=13,
    tiles=None
)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, HERE, Garmin, OpenStreetMap contributors",
    name="Peta Jalan",
    max_zoom=19
).add_to(peta)

for _, row in toko.iterrows():
    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=f"Toko yang sudah ada: {row['nama_toko']}",
        tooltip=row["nama_toko"],
        icon=folium.Icon(color="blue", icon="home")
    ).add_to(peta)

for _, row in kandidat.iterrows():
    warna = "green" if row["peringkat"] == 1 else "orange"

    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        tooltip=row["lokasi"],
        popup=f"""
        <b>{row['lokasi']}</b><br>
        Peringkat: {row['peringkat']}<br>
        Kepadatan penduduk: {row['kepadatan_penduduk']:,} orang/km²<br>
        Jarak ke toko terdekat: {row['jarak_toko_terdekat_km']} km<br>
        Skor potensi: {row['skor_potensi']}
        """,
        icon=folium.Icon(
            color=warna,
            icon="shopping-cart",
            prefix="fa"
        )
    ).add_to(peta)

display(peta)


from IPython.display import HTML, display

display(HTML("""
<div style="
    border: 1px solid #d1d5db;
    border-radius: 10px;
    padding: 18px;
    margin-top: 15px;
    color: #222;
    background-color: #ffffff;
    font-family: Arial, sans-serif;
">

    <h3 style="margin-top: 0; margin-bottom: 15px;">
        Keterangan Peta dan Hasil Analisis
    </h3>

    <table style="
        width: 100%;
        border-collapse: collapse;
        font-size: 14px;
    ">
        <tr style="background-color: #f1f5f9;">
            <th style="padding: 10px; text-align: left;">Penanda</th>
            <th style="padding: 10px; text-align: left;">Keterangan</th>
        </tr>

        <tr>
            <td style="padding: 10px; border-bottom: 1px solid #e5e7eb;">
                <span style="color: #2563eb;">●</span> Biru
            </td>
            <td style="padding: 10px; border-bottom: 1px solid #e5e7eb;">
                Lokasi toko yang sudah ada
            </td>
        </tr>

        <tr>
            <td style="padding: 10px; border-bottom: 1px solid #e5e7eb;">
                <span style="color: #16a34a;">●</span> Hijau
            </td>
            <td style="padding: 10px; border-bottom: 1px solid #e5e7eb;">
                Kandidat dengan skor potensi tertinggi
            </td>
        </tr>

        <tr>
            <td style="padding: 10px;">
                <span style="color: #f97316;">●</span> Oranye
            </td>
            <td style="padding: 10px;">
                Kandidat lokasi lainnya
            </td>
        </tr>
    </table>

    <h4 style="margin-bottom: 8px;">Cara Membaca Hasil</h4>
    <p style="line-height: 1.6; margin-top: 0;">
        Skor potensi dihitung berdasarkan kepadatan penduduk (60%)
        dan jarak dari toko terdekat (40%). Kandidat dengan skor
        tertinggi menjadi prioritas awal untuk ditinjau lebih lanjut.
    </p>

    <p style="
        font-size: 12px;
        color: #555;
        border-top: 1px solid #e5e7eb;
        padding-top: 10px;
        margin-bottom: 0;
    ">
        Catatan: Data lokasi dan kepadatan penduduk merupakan data simulasi.
        Jarak yang dihitung adalah jarak garis lurus, bukan jarak melalui jalan.
        Hasil analisis bukan rekomendasi bisnis final.
    </p>
</div>
"""))

Penanda,Keterangan
● Biru,Lokasi toko yang sudah ada
● Hijau,Kandidat dengan skor potensi tertinggi
● Oranye,Kandidat lokasi lainnya


In [24]:
terbaik = kandidat.iloc[0]
print("KANDIDAT LOKASI DENGAN SKOR TERTINGGI")
print(f"Lokasi: {terbaik['lokasi']}")
print(f"Kepadatan penduduk: {terbaik['kepadatan_penduduk']:,} orang/km²")
print(f"Jarak ke toko terdekat: {terbaik['jarak_toko_terdekat_km']} km")
print(f"Skor potensi: {terbaik['skor_potensi']}")
print("\nRekomendasi: jadikan lokasi ini kandidat untuk pemeriksaan lanjutan.")

KANDIDAT LOKASI DENGAN SKOR TERTINGGI
Lokasi: Kandidat 5
Kepadatan penduduk: 9,500 orang/km²
Jarak ke toko terdekat: 2.43 km
Skor potensi: 1.0

Rekomendasi: jadikan lokasi ini kandidat untuk pemeriksaan lanjutan.
